In [ ]:
#Full frame (Enhanced Version)
# 8 epoch 
#lr 1e-4
#bce

class unet_seq(BaseModel):    
    def __init__(self,bs,seq_len):
        super().__init__()

        self._6ab = torch.FloatTensor([0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1])
        self._289 = torch.FloatTensor([0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
        0., 1., 1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1.,
        1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
        
        self.vis = torch.tensor([0., 1., 1., 0., 1., 0., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 0., 1., 0.,
         0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0.,
         0., 1., 0., 0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1.]).cuda()
        self.lka = torch.zeros(88).cuda()
        
        self.history_reconstruct_conv = nn.Conv1d(1,200,(1,))
        
        self.lka_compressor = nn.Sequential(nn.Linear(88*3,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024*1,88),
                                            nn.Sigmoid())
        
        
        self.history_reconstruct = nn.Sequential(self.history_reconstruct_conv,
                                                 nn.Sigmoid())
        
        self.drop = nn.Dropout(0.1)
        self.previous_lka = torch.zeros(88).cuda()
        self.previous_states = torch.zeros(88).cuda()
        self.flag_list = []
        self.predection = []
        self.previous_states_conv = nn.Conv1d(2,1,(1))
        
        self.previous_vis = []
        self.state_list = []
        self.vis_value = 0
        
        self.c =0
        self.art_lka = []
        self.test = True
        self.max_count= 2352
        self.count = 0
        self.gen_vis = []
        
        
    def forward(self,x):   
        #torch.manual_seed(0)
        #self.lka = torch.zeros(88).cuda()
        input_list = x.view(80,-1).detach().cpu()
        vis_idx = np.where(((input_list[:,120:132] == self._6ab).sum(dim=1) // 12) == 1)[0]
        lka_idx = np.where(((input_list[:,120:208] == self._289).sum(dim=1) // 88) == 1)[0]
        if vis_idx.size and not self.test :
            self.count = (self.count+1)%836
            self.vis = x.view(80,-1)[vis_idx[0]][120:208]
            #self.vis_value = get_vis(self.vis[28:28+16].data)
            
            self.vis_value = self.gen_vis[self.count]            
            
            self.vis.data[28:28+8] = get_frame(self.vis_value)[:8].data.cuda()
            self.vis.data[28+12:28+16] = get_frame(self.vis_value)[-4:].data.cuda()
            self.vis_value = get_vis(self.vis[28:28+16].data)
            
        elif vis_idx.size:   
            self.count = (self.count+1)%836
            self.vis = x.view(80,-1)[vis_idx[0]][120:208]
            self.vis_value = get_vis(self.vis[28:28+16].data)
            if self.c > 2050 and self.c <2450:
            #    print(self.count)
            #    self.vis_value = self.vis_value - 1
                self.vis.data[28:28+8] = get_frame(self.vis_value)[:8].data.cuda()
                self.vis.data[28+12:28+16] = get_frame(self.vis_value)[-4:].data.cuda()
                
        if self.c==0 and not self.test:
            
            self.art_lka = []
            idx = random.randint(0,3)
            self.art_lka.extend( torch.randint(0,400,(1,(idx*2)+1)).tolist()[0])
            self.art_lka.extend( torch.randint(740,850,(1,random.randint(0,2))).tolist()[0])
            #self.art_lka.extend( torch.randint(510,570,(1,random.randint(0,2))).tolist()[0])
            print(self.art_lka)
            

        if self.c in self.art_lka:
            print('fake')
            self.lka =torch.logical_xor(self.lka.type(torch.BoolTensor),torch.ones_like(self.lka).type(torch.BoolTensor)).type(torch.FloatTensor).cuda()

        
        #Get the history of vis from previous vis and all the past frames
        history = self.previous_states_conv(torch.stack([self.previous_states,self.vis]).view(1,2,-1)).view(-1)
        history = self.drop(nn.functional.relu(history))
        predected_lka = self.lka_compressor(self.drop(torch.cat([self.previous_lka,history,self.vis])))
        
        
        if self.c > 200:
            predicted_hist = (self.history_reconstruct(history.view(1,1,-1)).view(200,-1))[:,28+4:28+16]
            actual_hist = (torch.stack(self.state_list[-200:]).view(200,-1))[:,28+4:28+16].cuda()
            
        else:
            predicted_hist = None
            actual_hist = None
            
        self.previous_lka = self.lka.detach().data
        self.previous_states = history.detach().data
        
        
        self.previous_vis.append(self.vis.detach().cpu().data)
        self.state_list.append(history.detach().cpu().data)
        
            
        
        self.flag_list.append([self.lka.detach().cpu(),self.vis.detach().cpu(),self.vis_value])
        self.c = (self.c + 1) % self.max_count
        return self.vis.type(torch.FloatTensor).cuda(),self.lka.type(torch.FloatTensor).cuda(),None,predected_lka,actual_hist,predicted_hist


In [ ]:
# Decimal input (Enahnced Version)
# 4 epoch 
#lr 1e-4
#mse
#parameter size = 1054746
#inference time = 7.576181157394465 ms/ mb

class unet_seq(BaseModel):    
    def __init__(self,bs,seq_len):
        super().__init__()

        self._6ab = torch.FloatTensor([0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1])
        self._289 = torch.FloatTensor([0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
        0., 1., 1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1.,
        1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
        
        self.vis = torch.tensor([0., 1., 1., 0., 1., 0., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 0., 1., 0.,
         0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0.,
         0., 1., 0., 0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1.]).cuda()
        self.lka = torch.zeros(1).cuda()
        
        self.history_reconstruct_conv = nn.Conv1d(1,10,(1,))
        
        #self.dec_lin = nn.Sequential(nn.Linear(1,88),
        #                                    nn.PReLU(),)
        
        self.lka_compressor = nn.Sequential(nn.Linear(3,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024*1,1),
                                            nn.Sigmoid())
        
        
        self.history_reconstruct = nn.Sequential(self.history_reconstruct_conv,
                                                 nn.Sigmoid())
        
        self.drop = nn.Dropout(0.1)
        self.previous_lka = torch.zeros(1).cuda()
        self.previous_states = torch.zeros(1).cuda()
        self.flag_list = []
        self.predection = []
        self.previous_states_conv = nn.Conv1d(2,1,(1))
        
        self.previous_vis = []
        self.state_list = []
        self.vis_value = 0
        
        self.c =0
        self.art_lka = []
        self.test = True
        self.max_count= 2352
        self.count = 0
        self.gen_vis = []
        
        
    def forward(self,x):   
        #torch.manual_seed(0)
        self.lka = torch.zeros(1).cuda()
        input_list = x.view(80,-1).detach().cpu()
        vis_idx = np.where(((input_list[:,120:132] == self._6ab).sum(dim=1) // 12) == 1)[0]
        lka_idx = np.where(((input_list[:,120:208] == self._289).sum(dim=1) // 88) == 1)[0]
        if vis_idx.size and not self.test :
            self.count = (self.count+1)%836            
            self.vis_value = self.gen_vis[self.count] /4           

            
        elif vis_idx.size:   
            self.count = (self.count+1)%836
            self.vis = x.view(80,-1)[vis_idx[0]][120:208]
            self.vis_value = get_vis(self.vis[28:28+16].data)/4
            
                
        if self.c==0 and not self.test:
            
            self.art_lka = []
            idx = random.randint(0,8)
            self.art_lka.extend( torch.randint(0,400,(1,(idx*2)+1)).tolist()[0])
            self.art_lka.extend( torch.randint(740,850,(1,random.randint(0,6))).tolist()[0])
            #self.art_lka.extend( torch.randint(510,570,(1,random.randint(0,2))).tolist()[0])
            print(self.art_lka)
            

        if self.c in self.art_lka:
            print('fake')
            self.lka =torch.logical_xor(self.lka.type(torch.BoolTensor),torch.ones_like(self.lka).type(torch.BoolTensor)).type(torch.FloatTensor).cuda()

        #Get the history of vis from previous vis and all the past frames
        self.vis_value = torch.tensor(self.vis_value).view(1,).type(torch.FloatTensor).cuda() 
        history = self.previous_states_conv(torch.stack([self.previous_states,self.vis_value]).view(1,2,-1)).view(-1)
        history = self.drop(nn.functional.relu(history))
        predected_lka = self.lka_compressor(self.drop(torch.cat([self.previous_lka,history,self.vis_value])))
        
        
        if self.c > 10:
            predicted_hist = (self.history_reconstruct(history.view(1,1,-1)).view(10,-1))
            actual_hist = (torch.stack(self.state_list[-10:]).view(10,-1)).cuda()
            
        else:
            predicted_hist = None
            actual_hist = None
            
        self.previous_lka = self.lka.detach().data
        self.previous_states = history.detach().data
        
        
        self.previous_vis.append(self.vis.detach().cpu().data)
        self.state_list.append(self.vis_value.detach().cpu().data)
        
            
        
        self.flag_list.append([self.lka.detach().cpu(),self.vis.detach().cpu(),self.vis_value])
        self.c = (self.c + 1) % self.max_count
        return self.vis.type(torch.FloatTensor).cuda(),self.lka.type(torch.FloatTensor).cuda(),None,predected_lka,actual_hist,predicted_hist


In [ ]:
#Predict lka full frame (basic without reconstruction)
# 8 epoch 
#lr 1e-4
#bce
class unet_seq(BaseModel):    
    def __init__(self,bs,seq_len):
        super().__init__()

        self._6ab = torch.FloatTensor([0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1])
        self._289 = torch.FloatTensor([0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
        0., 1., 1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1.,
        1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
        
        self.vis = torch.tensor([0., 1., 1., 0., 1., 0., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 0., 1., 0.,
         0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 1., 0.,
         0., 1., 0., 0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 0., 1.]).cuda()
        self.lka = torch.zeros(88).cuda()
        
        self.history_reconstruct_conv = nn.Conv1d(1,200,(1,))
        
        self.lka_compressor = nn.Sequential(nn.Linear(88*3,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024*1,88),
                                            nn.Sigmoid())
        
        
        self.history_reconstruct = nn.Sequential(self.history_reconstruct_conv,
                                                 nn.Sigmoid())
        
        self.drop = nn.Dropout(0.1)
        self.previous_lka = torch.zeros(88).cuda()
        self.previous_states = torch.zeros(88).cuda()
        self.flag_list = []
        self.predection = []
        self.previous_states_conv = nn.Conv1d(2,1,(1))
        
        self.previous_vis = []
        self.state_list = []
        self.vis_value = 0
        
        self.c =0
        self.art_lka = []
        self.test = True
        self.max_count= 2352
        self.count = 0
        self.gen_vis = []
    def forward(self,x):        
        #self.lka = torch.zeros(88).cuda()
        input_list = x.view(80,-1).detach().cpu()
        vis_idx = np.where(((input_list[:,120:132] == self._6ab).sum(dim=1) // 12) == 1)[0]
        lka_idx = np.where(((input_list[:,120:208] == self._289).sum(dim=1) // 88) == 1)[0]
        if vis_idx.size and not self.test :
            self.count = (self.count+1)%836
            self.vis = x.view(80,-1)[vis_idx[0]][120:208]
            #self.vis_value = get_vis(self.vis[28:28+16].data)
            
            self.vis_value = self.gen_vis[self.count]
            self.vis[28+4:28+16] = get_frame(self.vis_value).cuda()
        elif vis_idx.size:     
            self.vis = x.view(80,-1)[vis_idx[0]][120:208]
            self.vis_value = get_vis(self.vis[28:28+16].data)
            
        if self.c==0 and not self.test:
            
            self.art_lka = []
            idx = random.randint(0,3)
            self.art_lka.extend( torch.randint(0,430,(1,(idx*2)+1)).tolist()[0])
            self.art_lka.extend( torch.randint(730,850,(1,random.randint(0,2))).tolist()[0])
            self.art_lka.extend( torch.randint(1450,1600,(1,random.randint(0,2))).tolist()[0])
            print(self.art_lka)
            

        if self.c in self.art_lka:
            print('fake')
            self.lka =torch.logical_xor(self.lka.type(torch.BoolTensor),torch.ones_like(self.lka).type(torch.BoolTensor)).type(torch.FloatTensor).cuda()

        
        #Get the history of vis from previous vis and all the past frames
        history = self.previous_states_conv(torch.stack([self.previous_states,self.vis]).view(1,2,-1)).view(-1)
        history = self.drop(nn.functional.relu(history))
        predected_lka = self.lka_compressor(self.drop(torch.cat([self.previous_lka,history,self.vis])))
        
        
        
            
        self.previous_lka = self.lka.detach().data
        self.previous_states = history.detach().data
        
        
        self.previous_vis.append(self.vis.detach().cpu().data)
        self.state_list.append(history.detach().cpu().data)
        
            
        
        self.flag_list.append([self.lka.detach().cpu(),self.vis.detach().cpu(),self.vis_value])
        self.c = (self.c + 1) % self.max_count
        return None,self.lka.type(torch.FloatTensor).cuda(),None,predected_lka,None,None


One for both

In [1]:
import torch
import torch.nn as nn

In [31]:
class lka_predictor(nn.Module):    
    def __init__(self,bs,seq_len,rcn_w = 10,compress_hs=10,max_count = 131):
        super().__init__()
        self.bs,self.seq_len = bs,seq_len

        
        self.lka_pred = nn.Sequential(nn.Linear(1+seq_len+compress_hs,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024,1024*1),
                                            nn.PReLU(),
                                            nn.Dropout(0.4),
                                            nn.Linear(1024,1),
                                            nn.Sigmoid())
        
        self.rld_cmp = nn.Sequential(nn.Linear(seq_len+compress_hs,compress_hs),
                                            nn.PReLU())
        self.rld_rcn = nn.Sequential(nn.Linear(compress_hs,rcn_w*2))
        
        self.drop = nn.Dropout(p=0.4)
        
        #### LKA - Training - RLD Variables ###
        self.lka,self.mb_count,self.max_count,self.rcn_w  = None , 0, max_count,rcn_w
        self.p_lka = torch.zeros(self.bs,1).type(torch.float).cuda()
        self.rld = torch.tensor([0.5]*compress_hs).repeat(self.bs).view(self.bs,-1).cuda()
        #### Storage Elements####
        self.RLD_hist = []
        
    def forward(self,x):
        #########RLD COMPRESSION##########
        self.rld = self.drop(self.rld_cmp(torch.cat([x.squeeze(),self.rld],dim=-1)))
        print(self.rld.shape)
        ########END OF RLD COMPRESSION####
        if self.mb_count == 0:
            self.lka = [1]*50#generate_lka()
        #Get the history of vis from previous vis and all the past frames
        x_drop = self.drop(x).squeeze()
        print(self.p_lka.view(self.bs).shape,x_drop.shape,self.rld.shape)
        cat = torch.cat([self.p_lka,x_drop,self.rld],dim=-1)
        print(cat.shape)
        predicted_lka = self.lka_pred(cat)
        
        #### Updates for LKA - RLD history - Minibatch counter ####
        self.rld = self.rld.detach()
        self.p_lka = self.lka[self.mb_count: self.mb_count+self.bs].cuda()
        self.RLD_hist.extend(x.view(-1).detach().cpu().numpy())
        self.mb_count =(self.mb_count+self.bs)%self.max_count
        
        #### Reconstruction ####
        if len(self.RLD_hist) > self.rcn_w:
            rcn_rld = self.rld_rcn(self.rld).view(-1,2)
            tgt_rld = torch.tensor(self.RLD_hist[-self.rcn_w:]).clone().cuda()
        else:
            rcn_rld = self.rld_rcn(self.rld).view(-1,2)[:len(self.RLD_hist),:]
            tgt_rld = torch.tensor(self.RLD_hist).clone().cuda()
        return self.p_lka,predected_lka,(rcn_rld,tgt_rld)

In [32]:
m = lka_predictor(7,9).cuda()

In [33]:
m(torch.rand(7,9,1).cuda())

torch.Size([7, 10])
torch.Size([7]) torch.Size([7, 9]) torch.Size([7, 10])
torch.Size([7, 20])


AttributeError: 'list' object has no attribute 'cuda'

In [49]:
m.rld.shape

torch.Size([7, 10])

In [51]:
i=torch.rand(7,9,1).cuda()

In [58]:
torch.cat([torch.rand(7,9,1).squeeze().cuda(),m.rld],dim=-1).shape

torch.Size([7, 19])